# TFM: Análisis de Políticas de Sostenibilidad mediante técnicas de Argumentacion Computacional

## Clasificación de relaciones con google/flan-t5-base

Model page: https://huggingface.co/google/flan-t5-base

# No keywords

In [1]:
import os
import re
import math
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import gc

process_rel_path = r"/kaggle/working/TFM/Data/Relationships Keywords"
hf_model_repo = "google/flan-t5-base"
model_name = "flanT5"  

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(hf_model_repo)
model =AutoModelForSeq2SeqLM.from_pretrained(hf_model_repo).to(device)
model.eval()


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

2025-08-24 23:01:14.012122: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756076474.369998      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756076474.469660      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [2]:
!git clone https://github.com/camipalo/TFM.git

Cloning into 'TFM'...
remote: Enumerating objects: 2313, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 2313 (delta 50), reused 55 (delta 33), pack-reused 2239 (from 2)
Receiving objects: 100% (2313/2313), 90.61 MiB | 13.86 MiB/s, done.
Resolving deltas: 100% (1940/1940), done.
Updating files: 100% (1052/1052), done.


In [3]:
def preprocess_text(s):
    if not isinstance(s, str):
        return ""
    s = s.lower()
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    return s

def build_prompt(arg1, arg2):
    return (
        "Classify the relationship between the following two arguments.\n"
        "Respond with exactly one label from: Support, Attack, Rephrase, No Relationship.\n\n"

        f"Argument 1: {arg1}\n"
        f"Argument 2: {arg2}\n\n"
        "Label:"
    )

def normalize_label(text):
    t = (text or "").strip().lower()
    if "support" == t or t.startswith("support"):
        return "Support"
    if "attack" == t or t.startswith("attack") or "conflict" in t:
        return "Attack"
    if "rephrase" == t or "paraphrase" in t or t.startswith("reph"):
        return "Rephrase"
    if "no relationship" == t or "no relation" in t or "none" == t:
        return "No Relationship"
    # fallbacks for short/partial tokens
    if t in {"support", "attack", "rephrase"}:
        return t.capitalize()
    return "No Relationship"

def compute_max_length_for_prompts(input_dir, prefix_substring, tokenizer, safety_limit=None):   
    if safety_limit is None:
        safety_limit = min(getattr(tokenizer, "model_max_length", 512) or 512, 512)

    max_len = 0
    files = [f for f in os.listdir(input_dir) if f.endswith(".csv") and prefix_substring in f]
    for fn in files:
        path = os.path.join(input_dir, fn)
        df = pd.read_csv(path)
        if "SDGarg1" not in df.columns or "SDGarg2" not in df.columns:
            continue
        for a, b in zip(df["SDGarg1"], df["SDGarg2"]):
            a = preprocess_text(str(a) if pd.notna(a) else "")
            b = preprocess_text(str(b) if pd.notna(b) else "")
            prompt = build_prompt(a, b)
            ids = tokenizer(prompt, truncation=False, padding=False)["input_ids"]
            max_len = max(max_len, len(ids))
    return min(max_len if max_len > 0 else 128, safety_limit)


def predict_labels_flan(prompts, max_length):
    enc = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True, max_length=max_length
    ).to(device)

    outputs = model.generate(
        **enc,
        max_new_tokens=GEN_MAX_NEW_TOKENS,
        do_sample=False,              
        num_beams=1,
    )
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    labels = [normalize_label(x) for x in decoded]
    labels = [l if l in VALID_OUT else "No Relationship" for l in labels]
    return labels

def free_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def classify_relationships_flan(input_dir, prefix_substring):
    files = [f for f in os.listdir(input_dir) if f.endswith(".csv") and prefix_substring in f]
    if not files:
        print(f"No CSVs found in '{input_dir}' containing '{prefix_substring}'.")
        return

    # compute a safe prompt max length once per run
    MAX_LENGTH = compute_max_length_for_prompts(input_dir, prefix_substring, tokenizer)
    print(f"Using MAX_LENGTH = {MAX_LENGTH}")

    for fn in files:
        path = os.path.join(input_dir, fn)
        print(f"\nProcessing: {path}")
        df = pd.read_csv(path)

        if "SDGarg1" not in df.columns or "SDGarg2" not in df.columns:
            print(f"  Skipped (missing SDGarg1/SDGarg2): {fn}")
            continue

        df[rel_col] = ""

        n = len(df)
        total_done = 0
        first_examples = []

        batches = math.ceil(n / BATCH_SIZE)
        for bi in range(batches):
            s = bi * BATCH_SIZE
            e = min((bi + 1) * BATCH_SIZE, n)
            chunk = df.iloc[s:e]

            prompts = []
            a1_list = []
            a2_list = []
            for a1, a2 in zip(chunk["SDGarg1"], chunk["SDGarg2"]):
                a1 = preprocess_text(str(a1) if pd.notna(a1) else "")
                a2 = preprocess_text(str(a2) if pd.notna(a2) else "")
                prompts.append(build_prompt(a1, a2))
                a1_list.append(a1)
                a2_list.append(a2)

            try:
                labels = predict_labels_flan(prompts, max_length=MAX_LENGTH)
            except Exception as ex:
                print(f"  Batch {bi+1}/{batches} error: {ex}. Marking 'No Relationship'.")
                labels = ["No Relationship"] * len(prompts)

            df.loc[chunk.index, rel_col] = labels

            for a1, a2, lab in zip(a1_list, a2_list, labels):
                if len(first_examples) < 5:
                    first_examples.append((a1, a2, lab))
                elif len(first_examples) == 5:
                    print("Sample predictions (first 5):\n")
                    for a1, a2, lab in first_examples:
                        print(f"- Arg1: {a1[:]}\n-Arg2: {a2[:]}\nLabel: {lab}\n\n")
                    first_examples.append((a1, a2, lab))                                        

            total_done += len(prompts)
            if total_done % 100 < BATCH_SIZE: 
                print(f"  Progress: {total_done}/{n} relations classified...")

        # handle missing args (safety)
        mask_nan = df["SDGarg1"].isna() | df["SDGarg2"].isna()
        df.loc[mask_nan, rel_col] = "No Relationship"

        # Ensure valid set
        bad = ~df[rel_col].isin(VALID_OUT)
        if bad.any():
            df.loc[bad, rel_col] = "No Relationship"

        df.to_csv(path, index=False, encoding="utf-8")
        free_cuda()
        print(f"Saved: {path}")
        return df

## GLOBAL SDG 2023 

#### Qwen2.5 3B extraction

In [4]:
prefix = "intra_goalGLOBAL_SGD2023_qwen2.5-3b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
GEN_MAX_NEW_TOKENS = 3                
TEMPERATURE = 0.0
VALID_OUT = {"Support", "Attack", "Rephrase", "No Relationship"}


args_classified = classify_relationships_flan(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_flanT5"].value_counts()

Using MAX_LENGTH = 192

Processing: /kaggle/working/TFM/Data/Relationships Keywords/intra_goalGLOBAL_SGD2023_qwen2.5-3b.csv
Sample predictions (first 5):

- Arg1: at their core, the sdgs are an investment agenda: it is critical that un member states adopt and implement the sdg stimulus and support a comprehensive reform of the global financial architecture.
-Arg2: investing in statistical capacity, science, and data literacy are important priorities for achieving the sdgs.
Label: Support


- Arg1: at their core, the sdgs are an investment agenda: it is critical that un member states adopt and implement the sdg stimulus and support a comprehensive reform of the global financial architecture.
-Arg2: at the global level, averaging across countries, not a single sdg is currently projected to be met by 2030, with the poorest countries struggling the most.
Label: No Relationship


- Arg1: at their core, the sdgs are an investment agenda: it is critical that un member states adopt and impleme

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5
174,"The United States, as the world’s biggest econ...",The integration of the SDGs in national budget...,0_7,0_21,NaN,Support
19,"At their core, the SDGs are an investment agen...",A large majority of governments – 83 percent o...,0_0,0_20,NaN,Support
301,"First, that UN Member States, at the 2023 SDG ...",While comparable country-level data are not ye...,0_16,0_22,NaN,Support
212,All countries should use the half-way momentum...,"the SDG Index, including its regional and loca...",0_9,0_24,NaN,Support
34,"Investing in statistical capacity, science, an...",Most of the low-income and lower-middle income...,0_1,0_10,NaN,Support


rel_flanT5
Support            418
No Relationship     85
Attack               4
Name: count, dtype: int64

In [5]:
prefix = "cross_goalGLOBAL_SGD2023_qwen2.5-3b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
GEN_MAX_NEW_TOKENS = 3                
TEMPERATURE = 0.0
VALID_OUT = {"Support", "Attack", "Rephrase", "No Relationship"}


args_classified = classify_relationships_flan(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_flanT5"].value_counts()

Using MAX_LENGTH = 209

Processing: /kaggle/working/TFM/Data/Relationships Keywords/cross_goalGLOBAL_SGD2023_qwen2.5-3b.csv
Sample predictions (first 5):

- Arg1: at their core, the sdgs are an investment agenda: it is critical that un member states adopt and implement the sdg stimulus and support a comprehensive reform of the global financial architecture.
-Arg2: increased funding from the multilateral development banks (mdbs) and public development banks (pdbs) to low- and middle-income countries, linked to investments in the sdgs;
Label: Support


- Arg1: at their core, the sdgs are an investment agenda: it is critical that un member states adopt and implement the sdg stimulus and support a comprehensive reform of the global financial architecture.
-Arg2: many of them lack an adequately high sdg commitment, and almost all lack access to the necessary financial means to implement the sdgs.
Label: No Relationship


- Arg1: at their core, the sdgs are an investment agenda: it is critic

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5
679,While comparable country-level data are not ye...,"Inclusive economic growth, full and productive...",0_22,8_2,NaN,Support
1547,"the SDG Index, including its regional and loca...",Countries promote global cooperation for susta...,0_24,16_8,NaN,Support
2208,The pandemic and other crises have led to subs...,Sustainable cities: urban infrastructure and s...,3_1,11_0,NaN,Support
1909,Many of them lack an adequately high SDG commi...,"The European Green Deal (EGD), which is exempl...",1_1,9_1,NaN,Support
145,All countries should use the half-way momentum...,The pandemic and other crises have led to subs...,0_9,3_1,NaN,No Relationship


rel_flanT5
Support            2848
No Relationship    1110
Name: count, dtype: int64

#### Gemma3 27B extraction

In [6]:
prefix = "intra_goalGLOBAL_SGD2023_gemma3-27b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
GEN_MAX_NEW_TOKENS = 3                
TEMPERATURE = 0.0
VALID_OUT = {"Support", "Attack", "Rephrase", "No Relationship"}


args_classified = classify_relationships_flan(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_flanT5"].value_counts()

Using MAX_LENGTH = 230

Processing: /kaggle/working/TFM/Data/Relationships Keywords/intra_goalGLOBAL_SGD2023_gemma3-27b.csv
Sample predictions (first 5):

- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: despite this alarming development, the sdgs are still achievable.
Label: No Relationship


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: at their core, the sdgs are an investment agenda: it is critical that un member states adopt and implement the sdg stimulus and support a comprehensive reform of the global financial architecture.
Label: Support


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: to achieve the sdgs the world must both alter its current investment patterns and increase the overall volume of investments.
Label: Support


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: the stimulus’ urgent objectiv

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5
2600,Current geopolitical tensions are hindering SD...,Governments must take the lead in all six area...,16_1,16_14,NaN,No Relationship
1962,Unsustainable consumption is strongly intercon...,"Compared with other SDG monitoring reports, ho...",12_1,12_6,NaN,No Relationship
2492,the destruction of coastal wetland ecosystems;,"The Convention on Biological Diversity, adopte...",14_2,14_9,NaN,No Relationship
2653,to give all people the skills and knowledge to...,"There are no magic numbers, but rather a suite...",16_4,16_19,NaN,Support
2848,Achieving the SDGs requires global cooperation...,These fora are critical to encourage internati...,17_2,17_25,NaN,Support


rel_flanT5
Support            2406
No Relationship     860
Attack               88
Name: count, dtype: int64

In [7]:
prefix = "cross_goalGLOBAL_SGD2023_gemma3-27b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
GEN_MAX_NEW_TOKENS = 3                
TEMPERATURE = 0.0
VALID_OUT = {"Support", "Attack", "Rephrase", "No Relationship"}


args_classified = classify_relationships_flan(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_flanT5"].value_counts()

Using MAX_LENGTH = 241

Processing: /kaggle/working/TFM/Data/Relationships Keywords/cross_goalGLOBAL_SGD2023_gemma3-27b.csv
Sample predictions (first 5):

- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: greatly increased funding for national and subnational governments and private businesses in the emerging economies, especially the low-income countries (lics) and lower-middle-income countries (lmics), to carry out needed sdg actions;
Label: No Relationship


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: to give all people the skills and knowledge to end poverty, protect the environment, and build peaceful and inclusive societies.
Label: Support


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: although all governments are in principle committed to economic justice as enshrined in the universal declaration of human rights, and to the sdg tenets of ‘leave

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5
22747,"Unsustainable water management practices, incl...",Align private business investment flows with t...,6_2,8_1,NaN,No Relationship
31853,"Yet the SDG Dashboards rate rich countries, in...",Challenges such as decarbonization cannot be m...,12_5,13_14,NaN,No Relationship
20721,The HICs’ somewhat better performance on pilla...,The capacity and healthy functioning of ecosys...,4_15,14_6,NaN,Support
18630,"In HICs and LICs, the pandemic and other crise...","All countries, poorer and richer alike, should...",3_5,17_3,NaN,Support
15364,"Nonetheless, it is recognized as the birthplac...","Education builds human capital, which in turn ...",2_3,8_13,NaN,Support


rel_flanT5
Support            22406
No Relationship    13441
Attack               985
Name: count, dtype: int64

#### Gemma3 4B extraction

In [8]:
prefix = "intra_goalGLOBAL_SGD2023_gemma3-4b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
GEN_MAX_NEW_TOKENS = 3                
TEMPERATURE = 0.0
VALID_OUT = {"Support", "Attack", "Rephrase", "No Relationship"}


args_classified = classify_relationships_flan(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_flanT5"].value_counts()

Using MAX_LENGTH = 297

Processing: /kaggle/working/TFM/Data/Relationships Keywords/intra_goalGLOBAL_SGD2023_gemma3-4b.csv
Sample predictions (first 5):

- Arg1: the sdgs are seriously off track
-Arg2: the sdgs are still achievable
Label: No Relationship


- Arg1: the sdgs are seriously off track
-Arg2: it is critical that un member states adopt and implement the sdg stimulus
Label: Support


- Arg1: the sdgs are seriously off track
-Arg2: to achieve the sdgs the world must both alter its current investment patterns and increase the overall volume of investments
Label: No Relationship


- Arg1: the sdgs are seriously off track
-Arg2: revise the credit rating system and debt sustainability metrics to facilitate long-term sustainable development
Label: No Relationship


- Arg1: the sdgs are seriously off track
-Arg2: align private business investment flows with the sdgs, through improved national planning, regulation, reporting, and oversight
Label: No Relationship


  Progress: 250/6976

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5
2709,SDSN is also supporting new partnerships betwe...,"UN Member States, at the 2023 SDG Summit and t...",0_43,0_44,NaN,Support
4677,The COVID-19 pandemic also severely depleted t...,universal health coverage (UHC) is considered ...,3_3,3_10,NaN,No Relationship
671,"UN Member States should adopt an SDG Stimulus,...",For governments to combine the objectives of e...,0_8,0_36,NaN,Support
4602,"Sustainable ecosystems, sustainable agricultur...",Food systems are also highly vulnerable to cli...,2_21,2_25,NaN,Support
1134,UN Member States should commit to accelerating...,The SDGs are not only a public policy framewor...,0_14,0_64,NaN,Support


rel_flanT5
Support            4439
No Relationship    2489
Attack               48
Name: count, dtype: int64

In [9]:
prefix = "cross_goalGLOBAL_SGD2023_gemma3-4b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
GEN_MAX_NEW_TOKENS = 3                
TEMPERATURE = 0.0
VALID_OUT = {"Support", "Attack", "Rephrase", "No Relationship"}


args_classified = classify_relationships_flan(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_flanT5"].value_counts()

Using MAX_LENGTH = 317

Processing: /kaggle/working/TFM/Data/Relationships Keywords/cross_goalGLOBAL_SGD2023_gemma3-4b.csv
Sample predictions (first 5):

- Arg1: the sdgs are seriously off track
-Arg2: to achieve the sdgs the world must both alter its current investment patterns and increase the overall volume of investments.
Label: No Relationship


- Arg1: the sdgs are seriously off track
-Arg2: greatly increase funding to national and subnational governments and private businesses, especially in lics and lmics, to carry out needed sdg investments.
Label: Support


- Arg1: the sdgs are seriously off track
-Arg2: revise liquidity structures for lics and lmics, especially regarding sovereign debts, to forestall self-fulfilling banking and balance-of-payments crises;
Label: No Relationship


- Arg1: the sdgs are seriously off track
-Arg2: investing in statistical capacity, science, and data literacy are important priorities for achieving the sdgs
Label: No Relationship


- Arg1: the sdg

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5
33841,The SDGs incorporated from the start a strong ...,changes in natural land use configuration,1_30,15_2,NaN,No Relationship
29912,statistical capacity in the poorer and most vu...,Investments in research and development will a...,1_34,8_13,NaN,Support
37613,Today’s land-use practices and food systems ha...,"Expansion of private philanthropy, with focus ...",2_23,6_5,NaN,No Relationship
55978,globalized trade rules for ‘cleantech’ could a...,Achieving the SDGs requires more than ‘normal ...,8_6,13_12,NaN,No Relationship
71055,"To reduce inequalities, governments also need ...",The G20 should organize its financial cooperat...,16_29,17_13,NaN,Support


rel_flanT5
Support            39447
No Relationship    30309
Attack              1478
Name: count, dtype: int64

#### Llama3.3 70B extraction

In [10]:
prefix = "intra_goalGLOBAL_SGD2023_llama3.3-70b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
GEN_MAX_NEW_TOKENS = 3                
TEMPERATURE = 0.0
VALID_OUT = {"Support", "Attack", "Rephrase", "No Relationship"}


args_classified = classify_relationships_flan(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_flanT5"].value_counts()

Using MAX_LENGTH = 293

Processing: /kaggle/working/TFM/Data/Relationships Keywords/intra_goalGLOBAL_SGD2023_llama3.3-70b.csv
Sample predictions (first 5):

- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: since the outbreak of the pandemic in 2020 and other simultaneous crises, sdg progress has stalled globally.
Label: Attack


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: the world is off track, but that is all the more reason to double down on the sdgs.
Label: Support


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: at their core, the sdgs are an investment agenda: it is critical that un member states adopt and implement the sdg stimulus and support a comprehensive reform of the global financial architecture.
Label: Support


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: to achieve the sdgs the world must 

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5
5665,"Second, developed countries are not being held...",SDG 17 (Partnerships for the Goals) calls on a...,17_15,17_30,NaN,No Relationship
3720,While the high-income countries (HICs) and upp...,This is in sharp contrast to the pre-pandemic ...,10_10,10_14,NaN,No Relationship
3988,"Global spending on armaments, estimated at US$...","Yet the SDG Dashboards rate rich countries, in...",13_3,13_34,NaN,No Relationship
76,Since the outbreak of the pandemic in 2020 and...,To achieve the SDGs the world must both alter ...,0_1,0_4,NaN,Support
3013,1. Universal quality education and innovation-...,1. Universal quality education and innovation-...,4_3,4_7,NaN,Support


rel_flanT5
Support            4105
No Relationship    1674
Attack              172
Name: count, dtype: int64

In [11]:
prefix = "cross_goalGLOBAL_SGD2023_llama3.3-70b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
GEN_MAX_NEW_TOKENS = 3                
TEMPERATURE = 0.0
VALID_OUT = {"Support", "Attack", "Rephrase", "No Relationship"}


args_classified = classify_relationships_flan(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_flanT5"].value_counts()

Using MAX_LENGTH = 359

Processing: /kaggle/working/TFM/Data/Relationships Keywords/cross_goalGLOBAL_SGD2023_llama3.3-70b.csv
Sample predictions (first 5):

- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: the grim reality is that at the midpoint of the 2030 agenda, the sdgs are far off track.
Label: Support


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: similarly, extreme poverty can lead to a collapse of tax revenues, followed by government bankruptcy and further economic collapse, a syndrome that now threatens dozens of poor countries.
Label: Attack


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: most of the low-income and lower-middle income countries, home to more than the half of humanity, face major challenges in achieving most of the sdgs by 2030.
Label: Attack


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off trac

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5
22550,"According to IMF estimates in 2019, the financ...",Although all governments are in principle comm...,1_7,10_3,NaN,No Relationship
29074,Multiple and overlapping health and geopolitic...,The COVID-19 pandemic also severely depleted t...,3_6,16_5,NaN,No Relationship
21023,The SDGs are not only a public policy framewor...,"In 2022, the United Nations Secretary-General ...",0_68,17_3,NaN,Support
8736,UN Member States should commit to accelerating...,to ensure equitable access to and resources fo...,0_19,10_12,NaN,Support
18914,UN Member States must endorse a deep and overd...,"First, implementation is largely left to the n...",0_15,17_14,NaN,No Relationship


rel_flanT5
Support            33288
No Relationship    22996
Attack              2026
Name: count, dtype: int64

#### Deepseek r1 70B extraction

In [12]:
prefix = "intra_goalGLOBAL_SGD2023_deepseek-r1-70b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
GEN_MAX_NEW_TOKENS = 3                
TEMPERATURE = 0.0
VALID_OUT = {"Support", "Attack", "Rephrase", "No Relationship"}


args_classified = classify_relationships_flan(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_flanT5"].value_counts()

Using MAX_LENGTH = 280

Processing: /kaggle/working/TFM/Data/Relationships Keywords/intra_goalGLOBAL_SGD2023_deepseek-r1-70b.csv
Sample predictions (first 5):

- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: despite this alarming development, the sdgs are still achievable.
Label: No Relationship


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: the world is off track, but that is all the more reason to double down on the sdgs.
Label: Support


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: to achieve the sdgs the world must both alter its current investment patterns and increase the overall volume of investments.
Label: Support


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: all countries, poorer and richer alike, should use the half-way momentum to self-critically review and revise their national sdg strategi

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5
6259,"United Nations agencies, multilateral organiza...",Multilateralism and investments in global capa...,17_19,17_42,NaN,Support
4765,3. Zero-carbon energy systems: the transition ...,UNEP estimates that 84 percent of Parties to t...,13_0,13_22,NaN,Support
6471,Both sides – importers and exporters – must wo...,The SDGs require long-term directed change and...,17_32,17_33,NaN,Support
3769,"The interconnected environmental, social, and ...",Extreme poverty rates in LICs remain above pre...,3_1,3_6,NaN,No Relationship
5466,Failures of global governance Achieving the SD...,Nation-states continue to hold the primary res...,16_10,16_22,NaN,No Relationship


rel_flanT5
Support            4454
No Relationship    1864
Attack              219
Name: count, dtype: int64

In [13]:
prefix = "cross_goalGLOBAL_SGD2023_deepseek-r1-70b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
GEN_MAX_NEW_TOKENS = 3                
TEMPERATURE = 0.0
VALID_OUT = {"Support", "Attack", "Rephrase", "No Relationship"}


args_classified = classify_relationships_flan(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_flanT5"].value_counts()

Using MAX_LENGTH = 318

Processing: /kaggle/working/TFM/Data/Relationships Keywords/cross_goalGLOBAL_SGD2023_deepseek-r1-70b.csv
Sample predictions (first 5):

- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: the grim reality is that at the midpoint of the 2030 agenda, the sdgs are far off track.
Label: Support


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: at the global level, averaging across countries, not a single sdg is currently projected to be met by 2030, with the poorest countries struggling the most.
Label: Support


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: 1. increased funding from the multilateral develop-ment banks (mdbs) and public development banks (pdbs) to low- and middle-income countries, linked to investments in the sdgs;
Label: No Relationship


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5
46563,"According to the OECD, only one in 10 students...","To reduce inequalities, governments also need ...",9_15,10_20,NaN,Support
35851,The International Commission on the Futures of...,"For all of these reasons and more, neighboring...",4_8,9_9,NaN,Support
34832,Some countries specifically refer to the SDGs ...,Building on close cooperation with SDSN nation...,3_8,17_35,NaN,Support
51876,Disadvantaged communities have lower access to...,Achieving the SDGs requires more than 'normal ...,10_23,16_17,NaN,No Relationship
33341,"Those related to hunger, sustainable diets, an...","The European Green Deal (EGD), which is exempl...",3_5,12_1,NaN,Support


rel_flanT5
Support            34941
No Relationship    22855
Attack              3195
Name: count, dtype: int64